# Jump-Diffusion Research Notebook

This notebook explores a possible Phase 2 extension of the Probabilistic Price Path Engine.

The current dashboard supports:

- GBM Monte Carlo simulation
- historical bootstrap simulation
- analytical GBM terminal benchmarking
- pathwise TP/SL probability estimation

This notebook investigates whether a jump-diffusion model can better capture occasional large intraday moves.

The goal is research only. The model should be tested here before being added to the Streamlit dashboard.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

SRC_PATH = PROJECT_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from data_loader import load_market_data
from simulator import estimate_mu_sigma, compute_log_returns, simulate_gbm_paths, simulate_bootstrap_paths
from probability_engine import pathwise_tp_sl_metrics

In [ ]:
DATA_SOURCE = "synthetic"   # "synthetic", "mt5", or "csv"

SYMBOL = "US100.cash"
TIMEFRAME = "M1"
BARS = 1000

CSV_PATH = PROJECT_ROOT / "data" / "historical" / "sample.csv"

TP_POINTS = 29.0
SL_POINTS = 29.0

HORIZON = 20
N_PATHS = 10_000
VOL_WINDOW = 90

DRIFT_MODE = "zero"
DIRECTION = "long"

SEED = 42

In [ ]:
if DATA_SOURCE == "csv":
    df = load_market_data(
        source="csv",
        csv_path=CSV_PATH,
    )
else:
    df = load_market_data(
        source=DATA_SOURCE,
        symbol=SYMBOL,
        timeframe=TIMEFRAME,
        bars=BARS,
        synthetic_start_price=21500.0,
        synthetic_seed=SEED,
        synthetic_freq="1min",
    )

# Use closed-candle logic for MT5 research.
if DATA_SOURCE == "mt5" and len(df) > VOL_WINDOW + 21:
    model_df = df.iloc[:-1].copy()
else:
    model_df = df.copy()

model_df.tail()